# 11_ensemble_enhancement

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.base import clone

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
    BaggingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [2]:
# 设置文件夹路径
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "module_D_ensemble_ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("PROCESSED_DIR =", PROCESSED_DIR)
print("RESULTS_DIR =", RESULTS_DIR)

PROJECT_ROOT = d:\360Downloads\bioinformatics\Task\AIP
PROCESSED_DIR = d:\360Downloads\bioinformatics\Task\AIP\data\processed
RESULTS_DIR = d:\360Downloads\bioinformatics\Task\AIP\results\module_D_ensemble_ablation


In [3]:
# 读取文件
FEATURE_SET = "handcrafted_prott5_lasso"
FEATURE_PATH = PROCESSED_DIR / "fusion_selected" / "fusion_selected" /FEATURE_SET


X_train_full = np.load(FEATURE_PATH / "X_train.npy")
X_test = np.load(FEATURE_PATH / "X_test.npy")
y_train_full = np.load(FEATURE_PATH / "y_train.npy")
y_test = np.load(FEATURE_PATH / "y_test.npy")

print("X_train_full:", X_train_full.shape)
print("X_test:", X_test.shape)
print("y_train_full:", y_train_full.shape)
print("y_test:", y_test.shape)

X_train_full: (3583, 1209)
X_test: (897, 1209)
y_train_full: (3583,)
y_test: (897,)


In [4]:
# 划分验证集、训练集
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

print("X_tr:", X_tr.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_tr: (2866, 1209)
X_val: (717, 1209)
X_test: (897, 1209)


In [5]:
# 计算类别不平衡权重
n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())

scale_pos_weight = n_neg / max(n_pos, 1)

print("positive:", n_pos)
print("negative:", n_neg)
print("scale_pos_weight:", scale_pos_weight)

positive: 1092
negative: 1774
scale_pos_weight: 1.6245421245421245


In [6]:
# 写评估函数
def compute_binary_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        "ACC": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall_SN": sensitivity,
        "Specificity_SP": specificity,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    }
    return metrics


def evaluate_model(model, X, y, threshold=0.5):
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    metrics = compute_binary_metrics(y, y_pred, y_prob)
    return metrics, y_prob, y_pred

# 集成策略

In [7]:
scaler = StandardScaler()

X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled_full = scaler.transform(X_test)

print(X_tr_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

(2866, 1209) (717, 1209) (897, 1209)


In [8]:
base_models_extended = {
    "LGBM": {
        "model": LGBMClassifier(
            n_estimators=150,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "scaled": False
    },

    "XGBoost": {
        "model": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "scaled": False
    },

    "ExtraTrees": {
        "model": ExtraTreesClassifier(
            n_estimators=500,
            max_features="log2",
            min_samples_split=5,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "scaled": False
    },

    "RandomForest": {
        "model": RandomForestClassifier(
            n_estimators=400,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "scaled": False
    },

    "LogisticRegression": {
        "model": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "scaled": True
    },

    "SVM": {
        "model": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "scaled": True
    },

    "MLP": {
        "model": MLPClassifier(
            hidden_layer_sizes=(256, 64),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=300,
            random_state=RANDOM_STATE
        ),
        "scaled": True
    }
}

In [9]:
extended_val_results = {}
extended_val_probs = {}
trained_extended_models = {}

for name, item in base_models_extended.items():
    model = clone(item["model"])
    use_scaled = item["scaled"]

    if use_scaled:
        model.fit(X_tr_scaled, y_tr)
        metrics, y_prob, y_pred = evaluate_model(model, X_val_scaled, y_val)
    else:
        model.fit(X_tr, y_tr)
        metrics, y_prob, y_pred = evaluate_model(model, X_val, y_val)

    trained_extended_models[name] = model
    extended_val_results[name] = metrics
    extended_val_probs[name] = y_prob

    print(f"{name} finished.")

[LightGBM] [Info] Number of positive: 1092, number of negative: 1774
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030580 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21109
[LightGBM] [Info] Number of data points in the train set: 2866, number of used features: 1186
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
LGBM finished.
XGBoost finished.
ExtraTrees finished.
RandomForest finished.
LogisticRegression finished.
SVM finished.
MLP finished.


In [10]:
extended_val_df = pd.DataFrame(extended_val_results).T
extended_val_df = extended_val_df.sort_values(["MCC", "ROC_AUC", "ACC"], ascending=False)
extended_val_df

,ACC,Precision,Recall_SN,Specificity_SP,F1,MCC,ROC_AUC,PR_AUC,TP,TN,FP,FN
ExtraTrees,0.744770,0.684426,0.611722,0.826577,0.646035,0.449176,0.797033,0.697420,167.0,367.0,77.0,106.0
LGBM,0.737796,0.661597,0.637363,0.799550,0.649254,0.440211,0.768117,0.662668,174.0,355.0,89.0,99.0
RandomForest,0.741980,0.691304,0.582418,0.840090,0.632207,0.439520,0.781123,0.670671,159.0,373.0,71.0,114.0
SVM,0.736402,0.659091,0.637363,0.797297,0.648045,0.437594,0.747096,0.619195,174.0,354.0,90.0,99.0
XGBoost,0.730823,0.656250,0.615385,0.801802,0.635161,0.422798,0.759364,0.663753,168.0,356.0,88.0,105.0
MLP,0.700139,0.615079,0.567766,0.781532,0.590476,0.355256,0.724400,0.591556,155.0,347.0,97.0,118.0
LogisticRegression,0.652720,0.539216,0.604396,0.682432,0.569948,0.281587,0.703800,0.597484,165.0,303.0,141.0,108.0


In [14]:
strong_model_names = ["ExtraTrees", "RandomForest"]

strong_models = {}

for name in strong_model_names:
    strong_models[name] = base_models_extended[name]

strong_models

{'ExtraTrees': {'model': ExtraTreesClassifier(class_weight='balanced_subsample', max_features='log2',
                       min_samples_split=5, n_estimators=500, n_jobs=-1,
                       random_state=42),
  'scaled': False},
 'RandomForest': {'model': RandomForestClassifier(class_weight='balanced_subsample', n_estimators=400,
                         n_jobs=-1, random_state=42),
  'scaled': False}}

In [22]:
strong_soft_voting = VotingClassifier(
    estimators=[
        ("et", clone(base_models_extended["ExtraTrees"]["model"])),
        ("lgbm", clone(base_models_extended["LGBM"]["model"])),
    ],
    voting="soft",
    n_jobs=-1
)

In [23]:
# 用 full 数据训练
strong_soft_voting.fit(X_train_full_scaled, y_train_full)

strong_soft_metrics, strong_soft_prob, strong_soft_pred = evaluate_model(
    strong_soft_voting,
    X_test_scaled_full,
    y_test
)

pd.DataFrame([strong_soft_metrics], index=["Strong_SoftVoting"])

,ACC,Precision,Recall_SN,Specificity_SP,F1,MCC,ROC_AUC,PR_AUC,TP,TN,FP,FN
Strong_SoftVoting,0.746934,0.687296,0.616959,0.827027,0.650231,0.4545,0.794795,0.701033,211,459,96,131
